# Инструкция, выбор токенов и ответ по документу

Перед нами один документ о продлении книг. В нём есть срок и исключение, но нет числа продлений за семестр. Сначала сами определим ответы на три вопроса, затем сравним две инструкции для модели. Отдельные опыты покажут влияние декодирования и соблюдение JSON-формата.

In [ ]:
from pathlib import Path
import os
import json
import re
import unicodedata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Находим корень раздачи независимо от места запуска Jupyter.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "data" / "requests.csv").exists())
DATA = ROOT / "data"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
np.random.seed(42)
plt.rcParams.update({"font.family": "DejaVu Sans", "figure.figsize": (8, 4)})


## Читаем источник

Загрузим модель из подготовленного комплекта и откроем D03. Найдите в нём срок, условие чужой брони и сведения о числе продлений за семестр. Для последнего вопроса важно проверить именно наличие факта.

До вызова модели запишите ожидаемые ответы. Инструкция по подготовке файлов находится в START; в работе используется один и тот же источник и фиксированные веса.

In [ ]:
import math
import time
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed

torch.set_num_threads(4)
MODEL_ROOT = Path(os.environ.get("NLP_MODELS_DIR", ROOT / ".cache" / "models"))
MODEL_DIR = MODEL_ROOT / "generator"
if not (MODEL_DIR / "config.json").exists():
    raise FileNotFoundError("Сначала выполните scripts/prepare_models.py при доступном интернете")
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, local_files_only=True)
model, loading_info = AutoModelForCausalLM.from_pretrained(MODEL_DIR, local_files_only=True,
    dtype=torch.float32, output_loading_info=True)
if loading_info["missing_keys"] or loading_info["mismatched_keys"] or loading_info["error_msgs"]:
    raise RuntimeError("Не удалось полностью загрузить веса генератора")
model.eval(); model.requires_grad_(False)
assert next(model.parameters()).device.type == "cpu"
MAX_INPUT, MAX_NEW = 768, 64
documents = pd.read_csv(DATA / "knowledge.csv", keep_default_na=False)
document = documents.loc[documents["doc_id"].eq("D03"), "text"].iloc[0]
questions = {
    "duration": "На сколько дней можно продлить книгу?",
    "reservation": "Можно ли продлить книгу, если её забронировал другой читатель?",
    "count": "Сколько раз за семестр можно продлевать книгу?",
}
print(document)


### Один шаг выбора на числах

Для оценок (2;1;0) сначала вычислим обычный устойчивый softmax. Прибавьте к трём оценкам одно большое число и проверьте, изменились ли вероятности. Затем в задании добавим температуру и отбор кандидатов.

In [ ]:
demo_logits = np.array([2., 1., 0.])
demo_exp = np.exp(demo_logits - demo_logits.max())
print(demo_exp / demo_exp.sum())

## 1. Temperature и top-p

Напишите `temperature_probs(logits, T)` с T>0: поделите оценки на T и примените устойчивый softmax. При недопустимой температуре вызывайте `ValueError`. Создайте DataFrame `temperature_table` для оценок (2;1;0): индекс — 0.5, 1, 2; колонки — `token_A`, `token_B`, `token_C`.

Затем напишите `nucleus(probabilities, p)` для 0<p≤1. Верните массив исходной длины, оставив минимальный набор верхних вероятностей с суммой не меньше p. Остальные занулите, оставшиеся нормируйте. При неверных параметрах или ненормированном входе вызывайте `ValueError`.

Проверьте (0,5;0,3;0,2) при p=0,75. Что изменилось после исключения третьего токена? Почему greedy нельзя реализовать делением на T=0?

In [ ]:
def temperature_probs(logits, T):
    # Ваш код
    raise NotImplementedError

def nucleus(probabilities, p):
    # Ваш код
    raise NotImplementedError

# Ваш код
raise NotImplementedError

In [ ]:
assert np.allclose(temperature_table.sum(axis=1), 1)
assert np.allclose(temperature_probs([2,1,0],1), [0.66524096,0.24472847,0.09003057])
assert np.allclose(nucleus([0.5,0.3,0.2],.75), [.625,.375,0])
try:
    temperature_probs([1,2],0)
except ValueError:
    pass
else:
    raise AssertionError("Greedy не реализуется делением на T=0")

### Как сообщения превращаются во вход

Ниже небольшой пример штатного шаблона Qwen3. Он не вызывает генерацию: посмотрите, где находятся роли и начало ответа. Эту последовательность нужно сохранить в функции подготовки.

In [ ]:
demo_messages = [{"role": "system", "content": "Ты учебный помощник. Отвечай по-русски"},
                 {"role": "user", "content": "Что такое токен?"}]
demo_rendered = tokenizer.apply_chat_template(demo_messages, tokenize=False,
                                              add_generation_prompt=True, enable_thinking=False)
print(demo_rendered)

## 2. Подготовка входа

Закончите `prepare_input(user_text)`, возвращающую `(batch, input_length)`. Используйте такое же system-сообщение и параметры шаблона, как в рабочем примере. Второе сообщение имеет роль user и содержит переданный текст.

Полученную строку токенизируйте с `add_special_tokens=False`, `return_tensors="pt"`, без усечения. Для пустого или нестрокового входа, а также длины больше `MAX_INPUT`, вызывайте `ValueError`.

Выведите декодированный вход для короткого вопроса. Почему длину нужно проверять после шаблона, а добавление специальных токенов второй раз изменило бы запрос?

In [ ]:
def prepare_input(user_text):
    # Ваш код
    raise NotImplementedError


In [ ]:
b, n = prepare_input("Короткий учебный вопрос")
assert b["input_ids"].shape == b["attention_mask"].shape and b["input_ids"].shape[1] == n
assert 0 < n <= MAX_INPUT
try:
    prepare_input("слово "*2000)
except ValueError:
    pass
else:
    raise AssertionError("Большой вход должен быть отклонён, не усечён молча")

## Вызов генератора

Оболочка уже написана, чтобы основной опыт был посвящён ответам. Проследите в ней выделение новых токенов после `input_length`. Поля `text`, `prompt`, `input_tokens`, `new_tokens`, `stop` и параметры вызова понадобятся для сравнения и повторения опыта. Время хранится только среди технических сведений журнала.

In [ ]:
# Готовая оболочка генерации сохраняет метаданные и выделяет только новые токены.
def generate_answer(user_text, mode="greedy", temperature=0.7, top_p=0.8, seed=42, max_new=64):
    if mode not in {"greedy","sample"} or not 1 <= max_new <= MAX_NEW:
        raise ValueError("Проверьте режим и лимит")
    if mode == "sample" and (temperature <= 0 or not 0 < top_p <= 1):
        raise ValueError("Некорректные параметры выборки")
    batch, length = prepare_input(user_text)
    if length + max_new > model.config.max_position_embeddings:
        raise ValueError("Превышено окно модели")
    kwargs = {"max_new_tokens":max_new, "do_sample":mode=="sample",
              "pad_token_id":tokenizer.eos_token_id, "use_cache":True}
    if mode == "sample":
        kwargs.update(temperature=temperature, top_p=top_p, top_k=20)
    set_seed(seed)
    started = time.perf_counter()
    with torch.inference_mode():
        output = model.generate(**batch, **kwargs)
    new_ids = output[0, length:]
    eos = model.generation_config.eos_token_id
    eos_ids = {eos} if isinstance(eos, int) else set(eos)
    stopped = "EOS" if len(new_ids) and int(new_ids[-1]) in eos_ids else ("limit" if len(new_ids)>=max_new else "other")
    return {"text":tokenizer.decode(new_ids, skip_special_tokens=True), "prompt":user_text,
            "input_tokens":length, "new_tokens":len(new_ids), "stop":stopped,
            "seconds":time.perf_counter()-started, "mode":mode,
            "temperature":temperature if mode=="sample" else None,
            "top_p":top_p if mode=="sample" else None, "top_k":20 if mode=="sample" else None, "seed":seed}


## 3. Две инструкции к одному источнику

Напишите `build_prompt(question, variant)`, где variant — basic или strict. Оба варианта содержат один `document` и переданный вопрос. Basic содержит только документ, вопрос и начало ответа. Strict добавляет: «Используй только документ ниже. Ответь одним коротким предложением. Если нужного факта нет, ответь точно: НЕТ ДАННЫХ».

Функция `generate_answer` возвращает исходный ответ и сведения о вызове. Выполните шесть greedy-запросов: каждый вариант инструкции для каждого вопроса из `questions`. Сохраните список `prompt_records`; каждая запись содержит `question_id`, `variant`, `question` и все поля `generate_answer`. Создайте `prompt_results = pd.DataFrame(prompt_records)`.

Сначала сохраните ответы без исправлений. Их оценка по документу будет в последнем разделе.

In [ ]:
def build_prompt(question, variant):
    # Ваш код
    raise NotImplementedError

prompt_records = []
# Ваш код
raise NotImplementedError

In [ ]:
assert len(prompt_records) == 6
assert set(prompt_results["variant"]) == {"basic","strict"}
assert set(prompt_results["question_id"]) == set(questions)
assert prompt_results["mode"].eq("greedy").all()
assert prompt_results["input_tokens"].le(MAX_INPUT).all() and prompt_results["new_tokens"].between(1, MAX_NEW).all()

## 4. Один запрос, разные выборки

Возьмите strict-промпт для `questions["duration"]`. Выполните sampling при T=0.5 и 1.0, для каждой температуры seed 7 и 42. top_p=0.8; top_k=20 уже задан в оболочке. Сохраните четыре результата в `decoding_records` и DataFrame `decoding_results`.

Сравните их с greedy-ответом того же запроса из `prompt_results`. Если все строки совпали, оставьте это наблюдение в выводе. Как оно соотносится с расчётом распределений в первом задании?

In [ ]:
decoding_records = []
# Ваш код
raise NotImplementedError

In [ ]:
assert len(decoding_records) == 4
assert set(zip(decoding_results["temperature"], decoding_results["seed"])) == {(0.5,7),(0.5,42),(1.0,7),(1.0,42)}
assert decoding_results["prompt"].nunique() == 1 and decoding_results["top_p"].eq(.8).all()

## 5. Проверяем JSON

Напишите `validate_json_answer(text)`. Результат — словарь с `valid` (bool), `error` (str) и `parsed` (dict либо None). Принимайте только JSON-объект с ровно двумя ключами: `answer` — строка, `has_answer` — логическое значение. Markdown-ограждения и дополнительные поля не входят в требуемый формат. Ошибка возвращается в словаре и не прерывает разбор.

Отправьте отдельный greedy-запрос о сроке продления по D03, потребовав этот JSON без дополнительных слов. Сохраните ответ в `json_record`, проверку — в `json_validation`.

Сначала проверьте структуру, затем сопоставьте значение с документом. Почему успешный парсинг ещё не подтверждает ответ?

In [ ]:
def validate_json_answer(text):
    if not isinstance(text, str):
        return {"valid":False, "error":"Нужна строка", "parsed":None}
    # Ваш код
    raise NotImplementedError

# Ваш код
raise NotImplementedError

In [ ]:
assert validate_json_answer('{"answer":"14 дней","has_answer":true}')["valid"]
assert not validate_json_answer('{"answer":"нет","has_answer":"false"}')["valid"]
assert not validate_json_answer('```json\n{"answer":"14","has_answer":true}\n```')["valid"]
assert not validate_json_answer('{"answer":')["valid"]
assert type(json_validation["valid"]) is bool

## Сохраняем наблюдения

Следующая ячейка записывает исходные запросы и ответы в журнал. Он нужен, чтобы сравнение можно было повторить и чтобы последующий разбор не подменял формулировки модели.

In [ ]:
# Экспортируем исходные наблюдения, включая точные пользовательские промпты.
result_dir = ROOT / "outputs" / "L07"
result_dir.mkdir(parents=True, exist_ok=True)
records = {"system":"Ты учебный помощник. Отвечай по-русски", "checkpoint":"Qwen/Qwen3-0.6B",
           "prompt_runs":prompt_records, "decoding_runs":decoding_records,
           "json_run":json_record, "json_validation":json_validation}
(result_dir / "run_records.json").write_text(json.dumps(records, ensure_ascii=False, indent=2), encoding="utf-8")
print("Исходные наблюдения сохранены:", result_dir / "run_records.json")

## Разбор ответов

Для шести основных ответов составьте таблицу: вариант инструкции и вопрос; главный факт; формат strict; неподдержанные утверждения; отказ; пояснение по D03. Главный факт оценивайте от 0 до 2: 2 — верный ответ на вопрос, 1 — частичный без противоречия в главном факте, 0 — неверный, отсутствующий или отвечающий на другой вопрос.

Для strict формат получает 1 при выполнении требования короткой формы и точного «НЕТ ДАННЫХ», когда факта нет; иначе 0. Для basic пишите «не применимо». Содержательную уместность отказа обсуждайте отдельно от буквального совпадения строки.

Затем разберите четыре выборки и JSON. Совпали ли ответы при разных настройках? Подтверждает ли источник значение в корректном JSON? Если достигнут предел новых токенов, учитывайте возможную незавершённость ответа.

Сдайте ноутбук с этой таблицей и пояснениями. Перед сдачей перезапустите ядро и выполните все ячейки по порядку.